# Part 1: Learn the language of Clifford + T

In [93]:
from typing import Any

from kirin.ir import Method
import matplotlib.pyplot as plt
import numpy as np

from bloqade import squin, tsim, stim
from bloqade.cirq_utils import emit_circuit, load_circuit, noise
from bloqade.pyqrack import StackMemorySimulator
from bloqade.types import MeasurementResult, Qubit
from kirin.dialects.ilist import IList
import matplotlib.pyplot as plt

# this will help us have return types for our methods that have more intuitive names
Register = IList[Qubit, Any]
Measurement = IList[MeasurementResult, Any]

# this function will help us visualize some circuits
def show_circuit(squin_kernel):
    @squin.kernel
    def _to_visualize():
        _ = squin_kernel()

    return tsim.Circuit(_to_visualize).diagram(height=400)

# Part 2: Synthesize the rotation family

# Part 3: Non-Clifford gates are expensive

The optimal way is through a teleportation algorithm.
Parameters:
- Number of ancilla qubits = Nº of T in the 1 qubit circuit.
- Number of 2-qubit gates = Nº of CNOTs = Nº of ancilla qubits.
- Circuit depth = Each additional T gate adds a parallel subroutine that needs to be done. In the operations in the main qubit, only 1 2-qubit-gate is added per T. But if we now consider the process to create the |T> magic state, then the complexity is greater, since T gates are not fault tolerant normally. They should be created via purification.
- Feed forward: the more T you employ, you start to be limited by the time to measure the ancilla, which may dominate over the coherence time of the Rydberg atoms.


In [94]:
@squin.kernel
def t():
    qubits = squin.qalloc(1)
    squin.t(qubits[0])
    return qubits

show_circuit(t)

    


In [116]:



def create_circuit(gate_sequence):
    @squin.kernel
    def generated():
        qubits = squin.qalloc(2)
        for gate in gate_sequence:

            if gate == 'H':
                squin.h(qubits[0])

            elif gate == 'S':
                squin.s(qubits[0])

            elif gate == 'T':
                squin.h(qubits[1])
                squin.t(qubits[1])
                squin.cx(qubits[0], qubits[1])
                m = squin.broadcast.measure([qubits[1]])
                if m:
                    squin.s(qubits[0])
                squin.reset(qubits[1])
        return qubits
    return generated

circ = create_circuit(['H','S','T'])
show_circuit(circ)


InterpreterError: Interpreter EmitStimMain does not support New() using qubit dialect. Expected stim.gate, debug, lowering.call, stim.collapse, stim.noise, lowering.func, ssacfg, stim.aux, func

# Part 4: Move from one physical qubit to one logical qubit